Typical pre-processing steps needed:
1. Co-registration of 4 modalities - Done for all (close enough atlases)
2. Voxel spacing 1mm3 - done for all modalities
3. Skull-stripping - Done for TCGA, UCSF -but need to do for EGD (brain mask available)
4. N4 bias correction - tcga and EGD need bias correction.
5. Intensity normalization etc. is done in later steps with pyradiomics

This is very important - all downloaded scans are co-registered to a specific atlas. We don't have acccess to the non-coregistered images and segmentations would get messed up anyway.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install SimpleITK -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 24.4 MB/s eta 0:00:00


In [4]:
import os

In [5]:
TRIAL_DIR = '/content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_80'
DATA_DIR = '/content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_80/data'
PREPROCESSED_DIR = '/content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_80/preprocessed'

In [6]:
if not os.path.exists(PREPROCESSED_DIR):
  os.makedirs(PREPROCESSED_DIR)

In [7]:
cases = os.listdir(DATA_DIR)
tcga_cases = [case for case in cases if case.startswith('TCGA')]
ucsf_cases = [case for case in cases if case.startswith('UCSF')]
egd_cases = [case for case in cases if case.startswith('MR_EGD')]
utsw_cases = [case for case in cases if case.startswith('BT')]
print(f'Number of TCGA cases: {len(tcga_cases)}')
print(f'Number of UCSF cases: {len(ucsf_cases)}')
print(f'Number of MR_EGD cases: {len(egd_cases)}')
print(f'Number of UTSW cases: {len(utsw_cases)}')

Number of TCGA cases: 167
Number of UCSF cases: 501
Number of MR_EGD cases: 774
Number of UTSW cases: 625


In [8]:
import shutil

In [ ]:
# UCSF is all good just copy paste

import os
import nibabel as nib
import numpy as np
import SimpleITK as sitk


def preprocess_ucsf():
    for ucsf_case in ucsf_cases:
        print(ucsf_case)
        if "FU" in ucsf_case:
            continue
        print(f"\nProcessing Case: {ucsf_case}")
        case_out_dir = os.path.join(PREPROCESSED_DIR, ucsf_case)

        if ucsf_case != 'UCSF-PDGM-0182_nifti':
            continue

        # if os.path.exists(case_out_dir):
        #     print('Already done')
        #     continue

        # if not os.path.exists(case_out_dir):
        #     os.makedirs(case_out_dir)

        # 1. Load the Tumor Mask for this patient
        # Note: In BraTS, this is usually named 'seg.nii.gz'. Adjust if your file is named differently.
        tumor_mask_path = os.path.join(DATA_DIR, ucsf_case, 'seg.nii.gz')

        # Copy the original segmentation file to the preprocessed folder
        shutil.copy(tumor_mask_path, os.path.join(case_out_dir, 'seg.nii.gz'))
        tumor_mask_data = nib.load(tumor_mask_path).get_fdata()

        if os.path.exists(tumor_mask_path):
            tumor_mask_data = nib.load(tumor_mask_path).get_fdata()
            # Convert multi-class labels (1,2,4) into a single binary tumor mask
            binary_tumor_mask = (tumor_mask_data > 0).astype(np.uint8)
        else:
            print(f"  -> WARNING: No tumor mask found for {ucsf_case}. Skipping exclusion.")
            binary_tumor_mask = None

        # Only structural sequences! Do not add ADC or ASL to this list.
        sequences = ['flair.nii.gz', 't1ce.nii.gz', 't1.nii.gz', 't2.nii.gz']

        for sequence in sequences:
            sequence_path = os.path.join(DATA_DIR, ucsf_case, sequence)
            original_img = nib.load(sequence_path)
            sequence_data = original_img.get_fdata()

            # 2. Dynamically Generate the Brain Mask
            # Since BraTS files are already skull-stripped, any voxel > 0 is brain tissue.
            brain_mask_data = (sequence_data > 0).astype(np.uint8)

            # 3. Create the N4 "Weight Mask" (Healthy Tissue Only)
            if binary_tumor_mask is not None:
                healthy_tissue_mask = brain_mask_data - binary_tumor_mask
                # Clip to prevent negative values if there are any tiny mask mismatches
                healthy_tissue_mask = np.clip(healthy_tissue_mask, 0, 1)
            else:
                healthy_tissue_mask = brain_mask_data

            print(f"  -> Running Safe N4 Bias Correction on {sequence}...")

            # 4. SimpleITK Conversion
            sitk_img = sitk.GetImageFromArray(sequence_data.T)
            sitk_img = sitk.Cast(sitk_img, sitk.sitkFloat32)

            sitk_weight_mask = sitk.GetImageFromArray(healthy_tissue_mask.T.astype(np.uint8))

            # 5. Shrink Factor Optimization
            shrink_factor = 4
            shrunk_img = sitk.Shrink(sitk_img, [shrink_factor] * sitk_img.GetDimension())
            shrunk_mask = sitk.Shrink(sitk_weight_mask, [shrink_factor] * sitk_img.GetDimension())

            # 6. Execute N4 on the shrunk healthy tissue
            n4_filter = sitk.N4BiasFieldCorrectionImageFilter()
            n4_filter.Execute(shrunk_img, shrunk_mask)

            # 7. Apply the full-resolution bias field to the whole image
            log_bias_field = n4_filter.GetLogBiasFieldAsImage(sitk_img)
            corrected_sitk_img = sitk.DivideReal(sitk_img, sitk.Exp(log_bias_field))

            # 8. Save the corrected data
            corrected_data = sitk.GetArrayFromImage(corrected_sitk_img).T
            masked_img = nib.Nifti1Image(corrected_data, original_img.affine, original_img.header)

            sequence_base_name = sequence.split('.')[0]
            # Saving as _n4.nii.gz instead of _stripped since it was already stripped
            out_file_path = os.path.join(case_out_dir, f"{sequence_base_name}_n4.nii.gz")

            nib.save(masked_img, out_file_path)
            print(f"  -> Saved: {out_file_path}")



preprocess_ucsf()

UCSF-PDGM-0432_nifti

Processing Case: UCSF-PDGM-0432_nifti
UCSF-PDGM-0433_FU007d_nifti
UCSF-PDGM-0433_nifti

Processing Case: UCSF-PDGM-0433_nifti
UCSF-PDGM-0434_nifti

Processing Case: UCSF-PDGM-0434_nifti
UCSF-PDGM-0435_nifti

Processing Case: UCSF-PDGM-0435_nifti
UCSF-PDGM-0436_nifti

Processing Case: UCSF-PDGM-0436_nifti
UCSF-PDGM-0437_nifti

Processing Case: UCSF-PDGM-0437_nifti
UCSF-PDGM-0438_nifti

Processing Case: UCSF-PDGM-0438_nifti
UCSF-PDGM-0439_nifti

Processing Case: UCSF-PDGM-0439_nifti
UCSF-PDGM-0440_nifti

Processing Case: UCSF-PDGM-0440_nifti
UCSF-PDGM-0441_nifti

Processing Case: UCSF-PDGM-0441_nifti
UCSF-PDGM-0442_nifti

Processing Case: UCSF-PDGM-0442_nifti
UCSF-PDGM-0443_nifti

Processing Case: UCSF-PDGM-0443_nifti
UCSF-PDGM-0444_nifti

Processing Case: UCSF-PDGM-0444_nifti
UCSF-PDGM-0445_nifti

Processing Case: UCSF-PDGM-0445_nifti
UCSF-PDGM-0446_nifti

Processing Case: UCSF-PDGM-0446_nifti
UCSF-PDGM-0447_nifti

Processing Case: UCSF-PDGM-0447_nifti
UCSF-PDGM-04

In [ ]:
import os
import nibabel as nib
import numpy as np
import SimpleITK as sitk
import shutil


if not os.path.exists(PREPROCESSED_DIR):
    os.makedirs(PREPROCESSED_DIR)

# 1. Load Universal Brain Mask
brain_mask_path = '/content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_80/cohorts/egd/resources/PROJECT_DATA/Brain_mask.nii.gz'
brain_mask_data = nib.load(brain_mask_path).get_fdata()

def preprocess_egd():

    for egd_case in egd_cases:
        print(f"\nProcessing Case: {egd_case}")
        case_out_dir = os.path.join(PREPROCESSED_DIR, egd_case)
        if os.path.exists(case_out_dir):
            print('Already done')
            continue

        if not os.path.exists(case_out_dir):
            os.makedirs(case_out_dir)

        # 2. Load the specific Tumor Mask for this patient
        # (Assuming the mask is named 'tumor_mask.nii.gz' or similar in the case folder)
        tumor_mask_path = os.path.join(DATA_DIR, egd_case, 'seg.nii.gz')

        # Copy the original segmentation file to the preprocessed folder
        shutil.copy(tumor_mask_path, os.path.join(case_out_dir, 'seg.nii.gz'))
        tumor_mask_data = nib.load(tumor_mask_path).get_fdata()

        # 3. Create the N4 "Weight Mask" (Healthy Tissue Only)
        # Start with the whole brain, then subtract the tumor.
        # Now, 1 = healthy brain, 0 = background OR tumor.
        healthy_tissue_mask = brain_mask_data - tumor_mask_data

        # Ensure no negative values just in case masks don't perfectly overlap
        healthy_tissue_mask = np.clip(healthy_tissue_mask, 0, 1)

        sequences = ['flair.nii.gz', 't1ce.nii.gz', 't1.nii.gz', 't2.nii.gz']

        for sequence in sequences:
            sequence_path = os.path.join(DATA_DIR, egd_case, sequence)
            original_img = nib.load(sequence_path)
            sequence_data = original_img.get_fdata()

            # Apply standard brain mask to the image data
            masked_data = sequence_data * brain_mask_data

            print(f"  -> Running Safe N4 Bias Correction on {sequence}...")

            # Convert Image to SimpleITK
            sitk_img = sitk.GetImageFromArray(masked_data.T)
            sitk_img = sitk.Cast(sitk_img, sitk.sitkFloat32)

            # Convert our new HEALTHY TISSUE mask to SimpleITK
            sitk_weight_mask = sitk.GetImageFromArray(healthy_tissue_mask.T.astype(np.uint8))

            # Shrink factor optimization
            shrink_factor = 4
            shrunk_img = sitk.Shrink(sitk_img, [shrink_factor] * sitk_img.GetDimension())
            shrunk_mask = sitk.Shrink(sitk_weight_mask, [shrink_factor] * sitk_img.GetDimension())

            # Execute N4 using ONLY the healthy tissue to calculate the bias field
            n4_filter = sitk.N4BiasFieldCorrectionImageFilter()
            n4_filter.Execute(shrunk_img, shrunk_mask)

            # Extract the bias field and apply it to the FULL image (including the tumor)
            log_bias_field = n4_filter.GetLogBiasFieldAsImage(sitk_img)
            corrected_sitk_img = sitk.DivideReal(sitk_img, sitk.Exp(log_bias_field))

            # Save
            corrected_data = sitk.GetArrayFromImage(corrected_sitk_img).T
            masked_img = nib.Nifti1Image(corrected_data, original_img.affine, original_img.header)

            sequence_base_name = sequence.split('.')[0]
            out_file_path = os.path.join(case_out_dir, f"{sequence_base_name}_n4_stripped.nii.gz")
            nib.save(masked_img, out_file_path)
            print(f"  -> Saved: {out_file_path}")

preprocess_egd()


Processing Case: MR_EGD-0766
Already done

Processing Case: MR_EGD-0761
Already done

Processing Case: MR_EGD-0760
Already done

Processing Case: MR_EGD-0759
Already done

Processing Case: MR_EGD-0770
Already done

Processing Case: MR_EGD-0769
Already done

Processing Case: MR_EGD-0767
Already done

Processing Case: MR_EGD-0772
Already done

Processing Case: MR_EGD-0771
Already done

Processing Case: MR_EGD-0774
Already done

Processing Case: MR_EGD-0001
Already done

Processing Case: MR_EGD-0004
Already done

Processing Case: MR_EGD-0006
Already done

Processing Case: MR_EGD-0009
Already done

Processing Case: MR_EGD-0014
Already done

Processing Case: MR_EGD-0015
Already done

Processing Case: MR_EGD-0020
Already done

Processing Case: MR_EGD-0022
Already done

Processing Case: MR_EGD-0026
Already done

Processing Case: MR_EGD-0028
Already done

Processing Case: MR_EGD-0031
Already done

Processing Case: MR_EGD-0033
Already done

Processing Case: MR_EGD-0035
Already done

Processing

In [ ]:
import os
import nibabel as nib
import numpy as np
import SimpleITK as sitk

if not os.path.exists(PREPROCESSED_DIR):
    os.makedirs(PREPROCESSED_DIR)

def preprocess_tcga():

    for tcga_case in tcga_cases:
        print(f"\nProcessing Case: {tcga_case}")
        case_out_dir = os.path.join(PREPROCESSED_DIR, tcga_case)

        if not os.path.exists(case_out_dir):
            os.makedirs(case_out_dir)

        # 1. Load the Tumor Mask for this patient
        # Note: In BraTS, this is usually named 'seg.nii.gz'. Adjust if your file is named differently.
        tumor_mask_path = os.path.join(DATA_DIR, tcga_case, 'seg.nii.gz')

        # Copy the original segmentation file to the preprocessed folder
        shutil.copy(tumor_mask_path, os.path.join(case_out_dir, 'seg.nii.gz'))
        tumor_mask_data = nib.load(tumor_mask_path).get_fdata()

        if os.path.exists(tumor_mask_path):
            tumor_mask_data = nib.load(tumor_mask_path).get_fdata()
            # Convert multi-class labels (1,2,4) into a single binary tumor mask
            binary_tumor_mask = (tumor_mask_data > 0).astype(np.uint8)
        else:
            print(f"  -> WARNING: No tumor mask found for {tcga_case}. Skipping exclusion.")
            binary_tumor_mask = None

        # Only structural sequences! Do not add ADC or ASL to this list.
        sequences = ['flair.nii.gz', 't1ce.nii.gz', 't1.nii.gz', 't2.nii.gz']

        for sequence in sequences:
            sequence_path = os.path.join(DATA_DIR, tcga_case, sequence)
            original_img = nib.load(sequence_path)
            sequence_data = original_img.get_fdata()

            # 2. Dynamically Generate the Brain Mask
            # Since BraTS files are already skull-stripped, any voxel > 0 is brain tissue.
            brain_mask_data = (sequence_data > 0).astype(np.uint8)

            # 3. Create the N4 "Weight Mask" (Healthy Tissue Only)
            if binary_tumor_mask is not None:
                healthy_tissue_mask = brain_mask_data - binary_tumor_mask
                # Clip to prevent negative values if there are any tiny mask mismatches
                healthy_tissue_mask = np.clip(healthy_tissue_mask, 0, 1)
            else:
                healthy_tissue_mask = brain_mask_data

            print(f"  -> Running Safe N4 Bias Correction on {sequence}...")

            # 4. SimpleITK Conversion
            sitk_img = sitk.GetImageFromArray(sequence_data.T)
            sitk_img = sitk.Cast(sitk_img, sitk.sitkFloat32)

            sitk_weight_mask = sitk.GetImageFromArray(healthy_tissue_mask.T.astype(np.uint8))

            # 5. Shrink Factor Optimization
            shrink_factor = 4
            shrunk_img = sitk.Shrink(sitk_img, [shrink_factor] * sitk_img.GetDimension())
            shrunk_mask = sitk.Shrink(sitk_weight_mask, [shrink_factor] * sitk_img.GetDimension())

            # 6. Execute N4 on the shrunk healthy tissue
            n4_filter = sitk.N4BiasFieldCorrectionImageFilter()
            n4_filter.Execute(shrunk_img, shrunk_mask)

            # 7. Apply the full-resolution bias field to the whole image
            log_bias_field = n4_filter.GetLogBiasFieldAsImage(sitk_img)
            corrected_sitk_img = sitk.DivideReal(sitk_img, sitk.Exp(log_bias_field))

            # 8. Save the corrected data
            corrected_data = sitk.GetArrayFromImage(corrected_sitk_img).T
            masked_img = nib.Nifti1Image(corrected_data, original_img.affine, original_img.header)

            sequence_base_name = sequence.split('.')[0]
            # Saving as _n4.nii.gz instead of _stripped since it was already stripped
            out_file_path = os.path.join(case_out_dir, f"{sequence_base_name}_n4.nii.gz")

            nib.save(masked_img, out_file_path)
            print(f"  -> Saved: {out_file_path}")

preprocess_tcga()

In [29]:
import os
import nibabel as nib
import numpy as np
import SimpleITK as sitk

if not os.path.exists(PREPROCESSED_DIR):
    os.makedirs(PREPROCESSED_DIR)

def preprocess_utsw():

    for utsw_case in utsw_cases:
        print(f"\nProcessing Case: {utsw_case}")
        case_out_dir = os.path.join(PREPROCESSED_DIR, utsw_case)

        if not os.path.exists(case_out_dir):
            os.makedirs(case_out_dir)

        # 1. Load the Tumor Mask for this patient
        # Note: In BraTS, this is usually named 'seg.nii.gz'. Adjust if your file is named differently.
        tumor_mask_path = os.path.join(DATA_DIR, utsw_case, 'seg.nii.gz')

        # Copy the original segmentation file to the preprocessed folder
        shutil.copy(tumor_mask_path, os.path.join(case_out_dir, 'seg.nii.gz'))
        tumor_mask_data = nib.load(tumor_mask_path).get_fdata()

        if os.path.exists(tumor_mask_path):
            tumor_mask_data = nib.load(tumor_mask_path).get_fdata()
            # Convert multi-class labels (1,2,4) into a single binary tumor mask
            binary_tumor_mask = (tumor_mask_data > 0).astype(np.uint8)
        else:
            print(f"  -> WARNING: No tumor mask found for {utsw_case}. Skipping exclusion.")
            binary_tumor_mask = None

        # Only structural sequences! Do not add ADC or ASL to this list.
        sequences = ['flair.nii.gz', 't1ce.nii.gz', 't1.nii.gz', 't2.nii.gz']

        for sequence in sequences:
            sequence_path = os.path.join(DATA_DIR, utsw_case, sequence)
            original_img = nib.load(sequence_path)
            sequence_data = original_img.get_fdata()

            # 2. Dynamically Generate the Brain Mask
            # Since BraTS files are already skull-stripped, any voxel > 0 is brain tissue.
            brain_mask_data = (sequence_data > 0).astype(np.uint8)

            # 3. Create the N4 "Weight Mask" (Healthy Tissue Only)
            if binary_tumor_mask is not None:
                healthy_tissue_mask = brain_mask_data - binary_tumor_mask
                # Clip to prevent negative values if there are any tiny mask mismatches
                healthy_tissue_mask = np.clip(healthy_tissue_mask, 0, 1)
            else:
                healthy_tissue_mask = brain_mask_data

            print(f"  -> Running Safe N4 Bias Correction on {sequence}...")

            # 4. SimpleITK Conversion
            sitk_img = sitk.GetImageFromArray(sequence_data.T)
            sitk_img = sitk.Cast(sitk_img, sitk.sitkFloat32)

            sitk_weight_mask = sitk.GetImageFromArray(healthy_tissue_mask.T.astype(np.uint8))

            # 5. Shrink Factor Optimization
            shrink_factor = 4
            shrunk_img = sitk.Shrink(sitk_img, [shrink_factor] * sitk_img.GetDimension())
            shrunk_mask = sitk.Shrink(sitk_weight_mask, [shrink_factor] * sitk_img.GetDimension())

            # 6. Execute N4 on the shrunk healthy tissue
            n4_filter = sitk.N4BiasFieldCorrectionImageFilter()
            n4_filter.Execute(shrunk_img, shrunk_mask)

            # 7. Apply the full-resolution bias field to the whole image
            log_bias_field = n4_filter.GetLogBiasFieldAsImage(sitk_img)
            corrected_sitk_img = sitk.DivideReal(sitk_img, sitk.Exp(log_bias_field))

            # 8. Save the corrected data
            corrected_data = sitk.GetArrayFromImage(corrected_sitk_img).T
            masked_img = nib.Nifti1Image(corrected_data, original_img.affine, original_img.header)

            sequence_base_name = sequence.split('.')[0]
            # Saving as _n4.nii.gz instead of _stripped since it was already stripped
            out_file_path = os.path.join(case_out_dir, f"{sequence_base_name}_n4.nii.gz")

            nib.save(masked_img, out_file_path)
            print(f"  -> Saved: {out_file_path}")

preprocess_utsw()

Streaming output truncated to the last 5000 lines.

Processing Case: BT0122
  -> Running Safe N4 Bias Correction on flair.nii.gz...
  -> Saved: /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_80/preprocessed/BT0122/flair_n4.nii.gz
  -> Running Safe N4 Bias Correction on t1ce.nii.gz...
  -> Saved: /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_80/preprocessed/BT0122/t1ce_n4.nii.gz
  -> Running Safe N4 Bias Correction on t1.nii.gz...
  -> Saved: /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_80/preprocessed/BT0122/t1_n4.nii.gz
  -> Running Safe N4 Bias Correction on t2.nii.gz...
  -> Saved: /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_80/preprocessed/BT0122/t2_n4.nii.gz

Processing Case: BT0124
  -> Running Safe N4 Bias Correction on flair.nii.gz...
  -> Saved: /content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_80/preprocessed/BT0124/flair_n4.nii.gz
  -> Running Safe N4 Bias Correction on t1c

In [9]:
import os
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

# --- CONFIG ---
SEQUENCES = ['flair', 't1ce', 't1', 't2']   # matches your *_n4.nii.gz files
N_CASES_TO_SHOW = 10
QC_OUTPUT_DIR = os.path.join(PREPROCESSED_DIR, '_qc_overlays')

if not os.path.exists(QC_OUTPUT_DIR):
    os.makedirs(QC_OUTPUT_DIR)

# Collect candidate case folders (directories only, skip the QC dir itself)
all_entries = sorted(os.listdir(PREPROCESSED_DIR))
case_folders = []
for entry in all_entries:
    entry_path = os.path.join(PREPROCESSED_DIR, entry)
    if not os.path.isdir(entry_path):
        continue
    if entry.startswith('_'):
        continue
    case_folders.append(entry)

cases_to_show = case_folders[:N_CASES_TO_SHOW]
print(f"Will visualize {len(cases_to_show)} cases: {cases_to_show}")

# --- MAIN LOOP: one figure per case ---
for case_folder in cases_to_show:

    case_path = os.path.join(PREPROCESSED_DIR, case_folder)
    print(f"\nVisualizing: {case_folder}")

    # Load the mask first so we can pick the best slice
    mask_path = os.path.join(case_path, 'seg.nii.gz')
    if not os.path.exists(mask_path):
        print(f"  -> Skipping. No seg.nii.gz found.")
        continue

    mask_data = nib.load(mask_path).get_fdata()
    binary_mask = (mask_data > 0).astype(np.uint8)

    # Pick the axial slice (3rd axis) with the most tumor voxels.
    # If the mask is empty, fall back to the middle slice.
    tumor_voxels_per_slice = binary_mask.sum(axis=0).sum(axis=0)
    total_tumor_voxels = int(binary_mask.sum())

    if total_tumor_voxels > 0:
        best_slice_index = int(np.argmax(tumor_voxels_per_slice))
    else:
        best_slice_index = binary_mask.shape[2] // 2
        print(f"  -> WARNING: empty mask. Using middle slice {best_slice_index}.")

    print(f"  -> Tumor voxels: {total_tumor_voxels}. Showing slice {best_slice_index}.")

    # Prepare the figure: 2 rows (raw / overlay) x 4 columns (sequences)
    fig, axes = plt.subplots(2, len(SEQUENCES), figsize=(16, 8))
    fig.suptitle(f"{case_folder}  |  axial slice {best_slice_index}  |  tumor voxels {total_tumor_voxels}", fontsize=14)

    # Slice the mask once for overlay reuse
    mask_slice = binary_mask[:, :, best_slice_index]
    mask_slice_display = np.rot90(mask_slice)
    mask_slice_masked = np.ma.masked_where(mask_slice_display == 0, mask_slice_display)

    column_index = 0
    for seq in SEQUENCES:

        # Locate the preprocessed sequence file (e.g. flair_n4.nii.gz)
        seq_file_path = None
        for file_name in os.listdir(case_path):
            if file_name.startswith(f"{seq}_n4"):
                seq_file_path = os.path.join(case_path, file_name)
                break

        # If the sequence is missing, blank out both cells and move on
        if seq_file_path is None:
            print(f"  -> WARNING: missing {seq} sequence.")
            axes[0, column_index].set_title(f"{seq.upper()} (missing)")
            axes[0, column_index].axis('off')
            axes[1, column_index].axis('off')
            column_index = column_index + 1
            continue

        seq_data = nib.load(seq_file_path).get_fdata()
        seq_slice = seq_data[:, :, best_slice_index]
        seq_slice_display = np.rot90(seq_slice)

        # Robust contrast windowing from brain voxels (1st to 99th percentile)
        brain_values = seq_slice_display[seq_slice_display > 0]
        if brain_values.size > 0:
            vmin_value = np.percentile(brain_values, 1)
            vmax_value = np.percentile(brain_values, 99)
        else:
            vmin_value = seq_slice_display.min()
            vmax_value = seq_slice_display.max()

        # Top row: raw sequence
        axes[0, column_index].imshow(seq_slice_display, cmap='gray', vmin=vmin_value, vmax=vmax_value)
        axes[0, column_index].set_title(f"{seq.upper()}")
        axes[0, column_index].axis('off')

        # Bottom row: same sequence with tumor mask overlaid in red
        axes[1, column_index].imshow(seq_slice_display, cmap='gray', vmin=vmin_value, vmax=vmax_value)
        axes[1, column_index].imshow(mask_slice_masked, cmap='autumn', alpha=0.45, vmin=0, vmax=1)
        axes[1, column_index].set_title(f"{seq.upper()} + mask")
        axes[1, column_index].axis('off')

        column_index = column_index + 1

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # Save the figure, then display it inline
    out_png_path = os.path.join(QC_OUTPUT_DIR, f"{case_folder}_qc.png")
    plt.savefig(out_png_path, dpi=120, bbox_inches='tight')
    print(f"  -> Saved overlay: {out_png_path}")

    plt.show()
    plt.close(fig)

print("\nQC visualization complete.")

Output hidden; open in https://colab.research.google.com to view.